<h1 style="text-align: center;">Physical AI의 Vision-LLM 융합 시청각 멀티모달 시스템</h1>

<br><br>

<div style="text-align: right; color: gray; font-style: italic;">
강사 김규래&emsp;<br>
kkr.kyurae.kim@gmail.com&emsp;
</div><br>

---
---
<br>

## 0. Windows 개발 환경 초기 설정

### A. 한글 입력 설정

Windows 11 에는 한글 IME 가 기본 포함되어 있어 별도 설치가 필요 없습니다.

한글 입력이 되지 않는다면 아래를 확인하세요.

1. **설정 > 시간 및 언어 > 언어 및 지역** 이동
2. 언어 목록에 **한국어**가 없으면 `언어 추가` → `한국어` 선택
3. 한국어 우측 `...` → **언어 옵션** → 키보드에 **Microsoft IME** 가 있는지 확인
4. `한/영` 키 또는 `Alt` 키로 한/영 전환

> Jetson(Linux) 에서는 `ibus-hangul` 을 설치해야 했지만 Windows 에서는 불필요합니다.

### B. 프로젝트 폴더 생성

PowerShell 을 열고 작업할 위치에서 폴더를 만듭니다.

```powershell
mkdir vision-llm
cd vision-llm
```

> `mkdir` 은 PowerShell 에서도 그대로 동작합니다. (`New-Item -ItemType Directory` 의 별칭)

### C. Visual Studio Code 설치

https://code.visualstudio.com/download

해당 링크에서 **Windows x64 User Installer** 를 다운로드하여 실행합니다.

설치 중 **"PATH에 추가"** 와 **"Code(으)로 열기 작업을 탐색기 파일/디렉터리 상황에 맞는 메뉴에 추가"** 옵션을 체크하면 편리합니다.

설치가 끝나면 PowerShell 에서 확인합니다.

```powershell
code --version
```

> Jetson 에서는 `.deb (Arm64)` 패키지를 `sudo apt install` 로 설치했지만,
> Windows 에서는 `.exe` 설치 관리자를 실행하면 됩니다.

### D. Visual Studio Code Extensions 설치

1. Visual Studio Code 실행
2. 좌측 메뉴의 **Extensions** 선택
3. ***Python***과 ***Jupyter*** 설치

### E. Visual Studio Code 가상 환경 생성

1. Visual Studio Code 에서 ``Ctrl`` + ``Shift`` + ``` ` ``` 를 눌러 터미널(PowerShell) 열기

2. **Python 3.11 설치 확인**:
```powershell
py --list
```
목록에 `-V:3.11` 이 없다면 https://www.python.org/downloads/release/python-3119/ 에서
**Windows installer (64-bit)** 를 받아 설치하세요.

> **주의**: Python 3.12 이상(특히 3.13/3.14)에서는 `torch`, `mediapipe` 의 Windows 휠이
> 제공되지 않아 설치가 실패합니다. 반드시 **3.11** 을 사용하세요.

3. 가상 환경 생성:
```powershell
py -3.11 -m venv .venv
```

> Jetson 에서는 JetPack 이 제공하는 OpenCV/TensorRT 를 쓰기 위해
> `--system-site-packages` 옵션이 필요했지만, Windows 에서는 모든 패키지를
> pip 로 직접 설치하므로 이 옵션을 쓰지 않습니다.

4. 가상 환경 활성화:
```powershell
.\.venv\Scripts\Activate.ps1
```

만약 `이 시스템에서 스크립트를 실행할 수 없으므로` 오류가 나면 아래를 한 번 실행한 뒤 다시 시도하세요.
```powershell
Set-ExecutionPolicy -Scope CurrentUser -ExecutionPolicy RemoteSigned
```

5. `pip` 업그레이드:
```powershell
python -m pip install --upgrade pip setuptools wheel
```

6. 실습에 필요한 패키지 일괄 설치:
```powershell
pip install -r w_requirements.txt
```

7. Jupyter 커널 목록에 등록:
```powershell
python -m ipykernel install --user --name=vision_llm --display-name "Vision-LLM (Python3.11)"
```

8. VS Code 닫기 & 재실행

9. 우측 상단에 `Select Kernel` → `Jupyter Kernel...` → `Vision-LLM (Python3.11)` 선택

### F. GPU 및 CUDA 동작 확인

Jetson 에서는 실습 자원 확보를 위해 Chromium 을 삭제했지만, Windows 노트북에서는
필요하지 않습니다. 대신 **NVIDIA GPU 가 PyTorch 에서 보이는지** 먼저 확인합시다.

이 확인이 통과해야 03, 04 세션의 GPU 실습을 진행할 수 있습니다.

먼저 드라이버가 GPU 를 인식하는지 확인합니다.

```powershell
nvidia-smi
```

다음으로 가상 환경에서 PyTorch 가 CUDA 를 쓸 수 있는지 확인합니다.

```powershell
python -c "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))"
```

아래처럼 `True` 와 GPU 이름이 나오면 정상입니다.

```text
2.8.0+cu126 True NVIDIA GeForce RTX 3050 Ti Laptop GPU
```

`False` 가 나온다면 CPU 전용 torch 가 설치된 것이므로 재설치하세요.

```powershell
pip uninstall -y torch torchvision
pip install torch==2.8.0+cu126 torchvision==0.23.0+cu126 --extra-index-url https://download.pytorch.org/whl/cu126
```

> 별도의 CUDA Toolkit 설치는 필요 없습니다. PyTorch 휠에 CUDA 런타임이 포함되어 있으며,
> 최신 NVIDIA 그래픽 드라이버만 있으면 됩니다.

### G. Git 환경 설정

##### Git 최초 실행:

Windows 에서 Git 이 없다면 https://git-scm.com/download/win 에서 설치한 뒤 진행합니다.

```powershell
git config --global user.email "you@example.com"
```

또는

```powershell
git config --global user.name "Your Name"
```

##### 로그인 정보 저장:

```powershell
git config --global credential.helper manager
```

> Linux 에서는 `store`(평문 저장)를 썼지만, Windows 에서는 자격 증명 관리자에
> 안전하게 저장하는 `manager` 를 사용합니다.

---
---

<br><br><div style="text-align: right; color: gray; font-style: italic;">
© 2026, 김규래 (Kyu Rae Kim), All rights reserved.&emsp;<br><br>
This material is provided solely for the intended instructional purpose.&emsp;<br>
Redistribution, reproduction, modification, adaptation, or reuse of this material in any form without prior written permission from the copyright holder is prohibited.&emsp;
</div>